# 07 — Capstone: Synthetic Taxpayer Data Pipeline

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


**Optional consolidation exercise** (fit into practical time of modules 3–4, not additional scheduled JP). Read two sources, clean, join, validate, write Parquet.


In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


In [ ]:
tp=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/taxpayers.csv")
pay=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/payments.csv")
valid=pay.filter(F.col("status")=="PAID")
metrics=valid.groupBy("taxpayer_id").agg(F.sum("amount").alias("total_paid"),F.count("*").alias("payment_count"))
result=(tp.join(metrics,"taxpayer_id","left")
    .fillna({"total_paid":0,"payment_count":0}))
assert result.count()==tp.count(),"One row per taxpayer expected"
assert result.filter(F.col("total_paid")<0).count()==0
result.write.mode("overwrite").partitionBy("region").parquet("/home/spark/outputs/capstone_taxpayer_metrics")
result.groupBy("region").agg(F.sum("total_paid").alias("total_paid")).show()


### Submission checklist
Explain sources, cleaning, validation, join grain, output partitions, one performance consideration and one limitation of inference from simulated data.
